In [264]:
import numpy as np
from scipy.spatial import ConvexHull
from sklearn.preprocessing import StandardScaler
import pandas as pd
import utils.fdiv as fd

import networkx as nx
from scipy.spatial.distance import pdist, squareform

import matplotlib.pyplot as plt

import networkx as nx


In [248]:
import os
os.getcwd()

'/Users/qstuff/Desktop/Dummy_Data/FIA_eda'

In [265]:
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [266]:
df=pd.read_csv("data/FIA_ecological_traits.csv")

In [279]:
# sub_df = df[df['PID'] =="2_6_93_55328_501"]

sub_df1= df[df['PID'].str.startswith("6_6_7")]
sub_df1.drop(columns=['LON', 'LAT','PID', 'accepted_bin'], inplace=True)

sub_df1['abundances'] = sub_df1.groupby(list(sub_df1.columns)).transform('size')

sub_df1.drop_duplicates(inplace=True)

In [280]:
sub_df2= df[df['PID'].str.startswith("2_6_9")]
sub_df2.drop(columns=['LON', 'LAT','PID', 'accepted_bin'], inplace=True)

sub_df2['abundances'] = sub_df2.groupby(list(sub_df2.columns)).transform('size')

sub_df2.drop_duplicates(inplace=True)

In [281]:
abundance_df1 = pd.DataFrame()   
abundance_df1['species'] = sub_df1.pop('species')
abundance_df1['abundances'] = sub_df1.pop('abundances')


In [282]:
abundance_df2 = pd.DataFrame()   
abundance_df2['species'] = sub_df2.pop('species')
abundance_df2['abundances'] = sub_df2.pop('abundances')


In [283]:
trait_array1=np.array(sub_df1)
trait_array2=np.array(sub_df2)

In [270]:
def Relative_Abundance(df: pd.DataFrame, abundances: str):

    total = df[abundances].sum()
    df["Relative_Abundances"] = df[abundances] / total * 100
    return df   

In [277]:

def Functional_Richness(traits: np.ndarray) -> float:
    """
    Compute Functional Richness (FRic) as the volume of the convex hull
    in standardized trait space.

    Args:
        traits: np.ndarray of shape (S, T) where S = species, T = traits

    Returns:
        FRic (float) or None if not enough species to form a convex hull
    """
    # Standardize traits (mean=0, std=1)
    traits_scaled = StandardScaler().fit_transform(traits)

    n_species, n_traits = traits_scaled.shape
    if n_species <= n_traits:
        print("Not enough species to form a convex hull.")
        return None

    hull = ConvexHull(traits_scaled)
    FRic = hull.volume
    return FRic, hull

In [284]:
# abundance_df = Relative_Abundance(abundance_df, "abundances")

# r_ab = abundance_df['Relative_Abundances'].tolist()

# fd.Functional_Evenness(trait_array, r_ab)
_, hull1= Functional_Richness(trait_array1)
_, hull2 = Functional_Richness(trait_array2)
# fd.Functional_Divergence(trait_array, r_ab)
# fd.Raos_Q(trait_array, r_ab)

In [286]:

from scipy.spatial import Delaunay


def fric_intersect(hull1, hull2, n_samples: int = 100000) -> float:
    """
    Compute approximate Functional Volume Intersection (FRic_intersect) 
    between two communities using convex hulls.

    Args:
        hull1, hull2: scipy.spatial.ConvexHull objects for each community
        n_samples: number of Monte Carlo samples for approximation

    Returns:
        FRic_intersect: float between 0 and 1
    """
    # Combine points to get bounding box
    all_points = np.vstack([hull1.points, hull2.points])
    mins = all_points.min(axis=0)
    maxs = all_points.max(axis=0)

    # Generate random points in bounding box
    samples = np.random.uniform(mins, maxs, size=(n_samples, hull1.points.shape[1]))

    # Helper: check if points are inside a convex hull
    def in_hull(points, hull):
        delaunay = Delaunay(hull.points[hull.vertices])
        return delaunay.find_simplex(points) >= 0

    inside1 = in_hull(samples, hull1)
    inside2 = in_hull(samples, hull2)

    # Intersection fraction
    intersection_fraction = np.sum(inside1 & inside2) / n_samples
    union_fraction = np.sum(inside1 | inside2) / n_samples

    FRic_intersect = intersection_fraction / union_fraction if union_fraction > 0 else 0
    return FRic_intersect